# ¿Puentes ER, geometría del disco o MOND completo? El alcance λ en 3D

**Proyecto:** *Vacuum Particle Journey*, sexta parte.

**Lo que dijo el notebook 5:** la RAR con la masa extra repartida a escala **λ ≈ 0.66 kpc** mejora el ajuste de forma robusta, y **el haz del telescopio no lo explica** (el modelo instrumental pierde por ΔBIC ≈ 350–600; la componente física sobrevive sin los puntos interiores). λ **no depende de la densidad**: se comporta como una constante.

**Quedan dos explicaciones "normales" por descartar:**
1. **Geometría del disco:** los discos tienen grosor (cientos de pc) y la RAR se aplica como si toda la masa estuviera en el plano. Un suavizado de ~0.5 kpc podría estar compensando eso. Si es así, λ debería **crecer con la longitud de escala del disco R_d**.
2. **MOND completo (QUMOND) ya es no local.** La RAR que usamos es la fórmula algebraica $g = \nu(g_N/a_0)\,g_N$ aplicada punto a punto. Pero en la teoría completa, el campo sale de resolver una ecuación en todo el espacio:
$$\nabla^2\Phi = \nabla\cdot\big[\nu(|\nabla\Phi_N|/a_0)\,\nabla\Phi_N\big]$$
En un disco, esto se desvía de la fórmula algebraica. **Si esa desviación imita una λ de ~0.66 kpc, la señal es MOND y no hacen falta puentes ER.**

**Las tres pruebas de este notebook:**
- **A. Distancia:** ¿sale la misma λ en galaxias cercanas que en lejanas?
- **B. Disco:** ¿escala λ con R_d?
- **C. QUMOND en 3D:** se resuelve la gravedad MOND completa en una rejilla 3D para discos exponenciales de distintos tamaños, y se mide qué λ "aparente" produce.

Y las **visualizaciones 3D interactivas**: la masa fantasma de MOND alrededor de un disco (y, si aparecen, las regiones donde es *negativa*), una ilustración de la red de puentes ER con λ = 0.66 kpc, y la superficie de χ² de los datos reales.

> Tiempo en Colab: ~5–8 min. La parte 3D usa una rejilla de 128³ (`N_GRID`); con 96 va más rápido y con 160 es más precisa.

In [ ]:
import os, io, re, glob, zipfile, urllib.request, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import least_squares, minimize_scalar
from scipy.special import i0, i1, k0, k1
from scipy.stats import spearmanr

warnings.filterwarnings("ignore", category=RuntimeWarning)
RNG = np.random.default_rng(42)
QUICK = False

COL = {"Datos": "#222222", "Solo bariones": "#8a8a85", "NFW ΛCDM (c–M)": "#6250d6",
       "RAR (g† = 1.2e-10)": "#e87ba4", "RAR local (g† global)": "#eb6834", "RAR no local (ER)": "#1baf7a", "Instrumental": "#2a78d6", "Densidad": "#eda100", "Tamaño": "#e34948"}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7", "lines.linewidth": 2, "font.size": 10, "legend.frameon": False,
})

G = 4.30091e-6
C_KMS = 299792.458
H0 = 70.0; H0_KPC = H0 / 1000.0
KPC_M = 3.0857e19
TO_MS2 = 1e6 / KPC_M                    # (km/s)^2/kpc -> m/s^2
try:
    import plotly.graph_objects as go
    HAVE_PLOTLY = True
except ImportError:
    HAVE_PLOTLY = False
    print("plotly no disponible: las figuras 3D se harán con matplotlib (estáticas)")
from scipy import fft as sfft
N_GRID = 128

## 1. Datos SPARC, modelo RAR no local y distancias (de los notebooks 4 y 5)

In [ ]:
DATA_DIR = "sparc_rotmod"
BASES = ["https://astroweb.case.edu/SPARC/", "http://astroweb.cwru.edu/SPARC/"]
ZENODO = "https://zenodo.org/records/16284118/files/Rotmod_LTG.zip?download=1"

def _get(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    return urllib.request.urlopen(req, timeout=60).read()

def _files():
    return sorted(glob.glob(os.path.join(DATA_DIR, "**", "*_rotmod.dat"), recursive=True))

def get_sparc():
    os.makedirs(DATA_DIR, exist_ok=True)
    if _files():
        return True
    for url in [b + "Rotmod_LTG.zip" for b in BASES] + [ZENODO]:
        try:
            print("Descargando", url)
            zipfile.ZipFile(io.BytesIO(_get(url))).extractall(DATA_DIR)
            if _files():
                return True
        except Exception as e:
            print("  falló:", type(e).__name__)
    try:
        from google.colab import files
        print("Sube Rotmod_LTG.zip (de astroweb.case.edu/SPARC)")
        for name, content in files.upload().items():
            zipfile.ZipFile(io.BytesIO(content)).extractall(DATA_DIR)
        return bool(_files())
    except Exception:
        return False

def get_quality():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1")
            q = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    q[t[0]] = dict(Q=int(t[17]), Inc=float(t[5]))
            if len(q) > 100:
                return q
        except Exception:
            pass
    return None

def load_rotmod(path):
    name = os.path.basename(path).replace("_rotmod.dat", "")
    d = np.atleast_2d(np.loadtxt(path, comments="#"))
    g = dict(name=name, r=d[:, 0], vobs=d[:, 1], err=d[:, 2], vgas=d[:, 3], vdisk=d[:, 4], vbul=d[:, 5])
    ok = (g["r"] > 0) & (g["err"] > 0)
    o = np.argsort(g["r"][ok])
    for k in ["r", "vobs", "err", "vgas", "vdisk", "vbul"]:
        g[k] = g[k][ok][o]
    g["vgas2"] = g["vgas"] * np.abs(g["vgas"])
    g["vstar2"] = g["vdisk"] * np.abs(g["vdisk"]) + 1.4 * g["vbul"] * np.abs(g["vbul"])
    return g

def _v2_expdisk(r, M, Rd):
    y = r / (2 * Rd)
    return 4 * np.pi * G * (M / (2 * np.pi * Rd**2)) * Rd * y**2 * (i0(y) * k0(y) - i1(y) * k1(y))

def _v2_nfw(r, V200, c):
    x = r / (V200 / (10 * H0_KPC))
    mu = lambda y: np.log1p(y) - y / (1 + y)
    return V200**2 * mu(c * x) / (x * mu(c))

def make_synthetic_set(n=30, seed=0):
    rng = np.random.default_rng(seed)
    out = {}
    for name in ["NGC3198", "NGC2403", "DDO154"] + [f"SYN{i:03d}" for i in range(n - 3)]:
        V200 = 10 ** rng.uniform(1.6, 2.4); c = 10 ** rng.normal(1.0, 0.12)
        Rd = rng.uniform(1, 5); Ml = 10 ** rng.uniform(8.5, 11); Mg = Ml * 10 ** rng.uniform(-1, 0.7)
        r = np.linspace(0.3 * Rd, rng.uniform(5, 9) * Rd, rng.integers(12, 40))
        vdisk = np.sqrt(_v2_expdisk(r, Ml, Rd)); vgas = np.sqrt(_v2_expdisk(r, Mg, 2.5 * Rd))
        vtrue = np.sqrt(vgas**2 + 0.5 * vdisk**2 + _v2_nfw(r, V200, c))
        err = 2 + 0.04 * vtrue
        os.makedirs("synthetic", exist_ok=True)
        path = os.path.join("synthetic", f"{name}_rotmod.dat")
        np.savetxt(path, np.c_[r, vtrue + rng.normal(0, err), err, vgas, vdisk, 0 * r], header="Distance = 10 Mpc")
        out[name] = load_rotmod(path)
    return out

HAVE_SPARC = get_sparc()
if HAVE_SPARC:
    GAL = {}
    for f in _files():
        g = load_rotmod(f)
        if len(g["r"]) >= 5:
            GAL[g["name"]] = g
    Q = get_quality()
    if Q:
        GAL = {k: v for k, v in GAL.items() if k in Q and Q[k]["Q"] < 3 and Q[k]["Inc"] >= 30}
        print("Corte de calidad aplicado (Q<3, inc≥30°).")
    else:
        print("Sin tabla de calidad: se usan todas las galaxias.")
else:
    print("⚠️  SIN DATOS REALES: galaxias SINTÉTICAS con halo NFW. Los resultados no tienen valor físico.")
    GAL = make_synthetic_set()

if QUICK and len(GAL) > 40:
    keep = [k for k in ["NGC3198", "NGC2403", "DDO154"] if k in GAL]
    rest = [k for k in GAL if k not in keep]
    GAL = {k: GAL[k] for k in keep + list(RNG.choice(rest, 40 - len(keep), replace=False))}
N_TOT = sum(len(g["r"]) for g in GAL.values())
print(f"Galaxias: {len(GAL)} | puntos: {N_TOT}")

In [ ]:
NG = 400
U_GRID = np.geomspace(0.1, 1.5, 40)
LAM_GRID = np.r_[0.0, np.geomspace(0.1, 100, 24)]                                       # kpc
GD_GRID = np.unique(np.r_[np.geomspace(0.6e-10, 2.4e-10, 25), 1.2e-10]) / TO_MS2        # (km/s)^2/kpc
I_GD12 = int(np.argmin(np.abs(GD_GRID * TO_MS2 - 1.2e-10)))

def vb2(g, u):
    return g["vgas2"] + u * g["vstar2"]

def prior(u):
    return np.log10(u / 0.5) / 0.1

def rar_g(gb, gd):
    return gb / (-np.expm1(-np.sqrt(gb / gd)))

def F_matrix(r, a, lam):
    R, A = r[:, None], a[None, :]
    if lam == 0:
        return (A < R).astype(float)
    x = R / lam
    e_lo = np.exp(-np.clip((A - R) / lam, 0, None))
    e_hi = np.exp(-np.clip((R - A) / lam, 0, None))
    e_sum = np.exp(-(A + R) / lam)
    I_lo = e_lo * (x - 1) + e_sum * (1 + x)
    I_hi = 2 * A / lam - e_hi * (1 + x) + e_sum * (1 + x)
    return np.where(R <= A, I_lo, I_hi) * lam / (2 * A)

def setup(g):
    if "_s" in g:
        return g["_s"]
    rmax = g["r"].max()
    rg = np.linspace(rmax / NG, rmax, NG)
    s = dict(rg=rg, mids=0.5 * (np.r_[0.0, rg[:-1]] + rg),
             vg=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vgas2"]]),
             vs=np.interp(rg, np.r_[0.0, g["r"]], np.r_[0.0, g["vstar2"]]))
    g["_s"] = s
    return s

def phantom_shells(s, U, GD):
    # G·M_fantasma en cascarones -> forma (nΥ, ng†, NG)
    v2 = s["vg"][None, :] + U[:, None] * s["vs"][None, :]
    gb = np.clip(v2, 1e-6, None) / s["rg"][None, :]
    gr = rar_g(gb[:, None, :], GD[None, :, None])
    GM = s["rg"][None, None, :] ** 2 * (gr - gb[:, None, :])
    return np.diff(np.concatenate([np.zeros(GM.shape[:2] + (1,)), GM], axis=2), axis=2)

def model_v2(g, u, lam, gd):
    s = setup(g)
    d = phantom_shells(s, np.array([u]), np.array([gd]))[0, 0]
    return vb2(g, u) + F_matrix(g["r"], s["mids"], lam) @ d / g["r"]

# Comprobaciones: F(λ→0) es un escalón; con λ pequeño se recupera la RAR local exacta
_g = next(iter(GAL.values()))
v_loc = np.sqrt(_g["r"] * rar_g(np.clip(vb2(_g, 0.5), 1e-6, None) / _g["r"], 1.2e-10 / TO_MS2))
v_0 = np.sqrt(model_v2(_g, 0.5, 0.0, 1.2e-10 / TO_MS2))
v_s = np.sqrt(model_v2(_g, 0.5, 1e-3, 1.2e-10 / TO_MS2))
print(f"Test: |v(λ=0) − RAR exacta| máx = {np.max(np.abs(v_0 - v_loc)):.2f} km/s; "
      f"|v(λ=0.001) − v(λ=0)| máx = {np.max(np.abs(v_s - v_0)):.3f} km/s (ambos deben ser ≈ 0)")

In [ ]:
ARCSEC = 206265.0
paths = _files() if HAVE_SPARC else glob.glob(os.path.join("synthetic", "*_rotmod.dat"))
DIST = {}
for f in paths:
    with open(f) as fh:
        for line in fh:
            m = re.search(r"Distance\s*=\s*([\d.]+)", line)
            if m:
                DIST[os.path.basename(f).replace("_rotmod.dat", "")] = float(m.group(1)); break
if not HAVE_SPARC:
    _rd = np.random.default_rng(3)
    DIST = {k: float(10 ** _rd.uniform(0.5, 1.8)) for k in GAL}
    print("⚠️  Distancias sintéticas aleatorias (solo para probar el código)")
GAL = {k: g for k, g in GAL.items() if k in DIST}
NAMES = list(GAL)
D = np.array([DIST[k] for k in NAMES])
RMAX = np.array([GAL[k]["r"].max() for k in NAMES])
MB = np.array([max(GAL[k]["r"][-1] * vb2(GAL[k], 0.5)[-1] / G, 1e5) for k in NAMES])
SIGMA = MB / (np.pi * (RMAX * 1e3) ** 2)          # M☉/pc²
N_TOT_PTS = sum(len(g["r"]) for g in GAL.values())
print(f"{len(NAMES)} galaxias | D: {D.min():.1f}–{D.max():.1f} Mpc | Σ_b: {np.median(SIGMA):.1f} M☉/pc² (mediana)")
rho, p = spearmanr(np.log10(D), np.log10(RMAX))
print(f"Correlación distancia–tamaño en la muestra: ρ = {rho:.2f} (p = {p:.0e})  ← por esto λ instrumental imitaría 'λ ∝ tamaño'")

## 2. Longitud de escala del disco R_d

Se toma de la tabla general de SPARC si se puede descargar. Si no, se estima a partir del máximo de la curva del disco estelar: en un disco exponencial, $V_{disk}$ es máxima en $r \approx 2.2\,R_d$.

In [ ]:
def get_rdisk():
    for b in BASES:
        try:
            txt = _get(b + "SPARC_Lelli2016c.mrt").decode("latin-1"); out = {}
            for line in txt.splitlines():
                t = line.split()
                if len(t) >= 18 and re.fullmatch(r"\d+", t[1] or "") and t[17] in ("1", "2", "3"):
                    out[t[0]] = float(t[11])
            if len(out) > 100:
                return out
        except Exception:
            pass
    return {}

RD_TAB = get_rdisk()
RD = []
for k in NAMES:
    g = GAL[k]
    est = g["r"][np.argmax(np.abs(g["vdisk"]))] / 2.2 if np.any(g["vdisk"] != 0) else np.nan
    RD.append(RD_TAB.get(k, est) if RD_TAB.get(k, 0) > 0 else est)
RD = np.array(RD)
RD = np.where(np.isfinite(RD) & (RD > 0), RD, np.nanmedian(RD))
print(f"R_d {'de la tabla de SPARC' if RD_TAB else 'estimado desde V_disk'}: mediana {np.median(RD):.2f} kpc "
      f"(16–84 %: {np.percentile(RD,16):.2f}–{np.percentile(RD,84):.2f})")

## 3. Tablas de χ² (con 5 % sistemático; y además sin los puntos interiores)

In [ ]:
LAM_GRID = np.r_[0.0, np.geomspace(0.02, 200, 45)]
LOGL = np.log(np.r_[0.01, LAM_GRID[1:]])
BEAM_ARCSEC = 15.0
LAM_GRID = np.r_[0.0, np.geomspace(0.02, 200, 45)]
LOGL = np.log(np.r_[0.01, LAM_GRID[1:]])
BEAM_ARCSEC = 15.0

def chi_table(floor=0.0, cut_beams=None):
    TOT = np.zeros((len(NAMES), len(LAM_GRID), len(GD_GRID))); DAT = np.zeros_like(TOT); NPT = np.zeros(len(NAMES), int)
    for n_i, name in enumerate(NAMES):
        g = GAL[name]; s = setup(g)
        err = np.sqrt(g["err"] ** 2 + (floor * g["vobs"]) ** 2)
        mask = np.ones(len(g["r"]), bool)
        if cut_beams:
            mask = g["r"] >= cut_beams * BEAM_ARCSEC / ARCSEC * D[n_i] * 1e3
            if mask.sum() < 5:
                mask = np.zeros(len(g["r"]), bool); mask[-5:] = True
        NPT[n_i] = mask.sum()
        d = phantom_shells(s, U_GRID, GD_GRID)
        base = vb2(g, U_GRID[:, None]); pr = prior(U_GRID) ** 2
        for l_i, lam in enumerate(LAM_GRID):
            extra = np.einsum("in,ugn->ugi", F_matrix(g["r"], s["mids"], lam), d) / g["r"]
            v = np.sqrt(np.clip(base[:, None, :] + extra, 0, None))
            c = ((((g["vobs"] - v) / err) ** 2) * mask).sum(-1)
            t = c + pr[:, None]; k = t.argmin(0); a = np.arange(len(GD_GRID))
            TOT[n_i, l_i] = t[k, a]; DAT[n_i, l_i] = c[k, a]
    return dict(tot=TOT, dat=DAT, npts=NPT)


t0 = time.time()
TAB5 = chi_table(floor=0.05)
TABC = chi_table(floor=0.05, cut_beams=2)
print(f"Tablas en {time.time()-t0:.0f} s")

def per_gal(tab, lam_vec, which="tot", idx=None):
    T = tab[which] if idx is None else tab[which][idx]
    n = np.arange(T.shape[0])
    ll = np.log(np.clip(lam_vec, 0.01, 200))
    k = np.clip(np.searchsorted(LOGL, ll), 1, len(LOGL) - 1)
    w = np.clip((ll - LOGL[k - 1]) / (LOGL[k] - LOGL[k - 1]), 0, 1)[:, None]
    return T[n, k - 1, :] * (1 - w) + T[n, k, :] * w

def lam_profile(tab, idx):
    S = tab["tot"][idx].sum(0)            # λ, g†
    return S.min(1)

## 4. Prueba A: ¿la misma λ cerca y lejos?

Se divide la muestra en tres grupos por distancia y se ajusta λ en cada uno, con bootstrap de galaxias dentro del grupo. Si el haz del telescopio fuera la causa, λ crecería con la distancia (en un grupo a 30 Mpc, 15″ son 2.2 kpc; a 5 Mpc, 0.36 kpc).

In [ ]:
edges = np.percentile(D, [0, 33.3, 66.7, 100]); edges[-1] += 1
rowsA = []
for lo, hi in zip(edges[:-1], edges[1:]):
    idx = np.where((D >= lo) & (D < hi))[0]
    for label, tab in [("5 %", TAB5), ("5 % + sin interior", TABC)]:
        prof = lam_profile(tab, idx); lb = LAM_GRID[np.argmin(prof)]
        bl = []
        for _ in range(300):
            sub = RNG.choice(idx, len(idx)); bl.append(LAM_GRID[np.argmin(lam_profile(tab, sub))])
        bl = np.array(bl)
        rowsA.append(dict(grupo=f"{lo:.1f}–{hi:.1f} Mpc", galaxias=len(idx), condiciones=label, λ_kpc=round(lb, 3),
                          IC68=f"{np.percentile(bl,16):.2f}–{np.percentile(bl,84):.2f}",
                          haz_15as_kpc=round(15 / ARCSEC * np.median(D[idx]) * 1e3, 2),
                          Δχ2_vs_local=round(prof[0] - prof.min(), 1)))
TA = pd.DataFrame(rowsA)
print(TA.to_string(index=False))

## 5. Prueba B: ¿escala λ con el disco?

Competición global, igual que en el notebook 5: λ constante frente a λ = α·R_d (proporcional al disco) y frente a λ = λ₀ (R_d/3 kpc)^γ.

In [ ]:
def compete_B(tab):
    npts = tab["npts"].sum(); rows = []
    cands = {
        "λ constante": ([dict(L=L) for L in np.geomspace(0.02, 50, 80)], lambda p: np.full(len(NAMES), p["L"]), 1),
        "λ = α·R_d": ([dict(a=a) for a in np.geomspace(0.005, 10, 80)], lambda p: p["a"] * RD, 1),
        "λ = λ₀(R_d/3)^γ": ([dict(L=L, gm=gm) for L in np.geomspace(0.02, 50, 40) for gm in np.linspace(-1.5, 2, 36)],
                            lambda p: p["L"] * (RD / 3) ** p["gm"], 2),
    }
    for m, (grid, f, k) in cands.items():
        bt, bp, bj = np.inf, None, None
        for p in grid:
            S = per_gal(tab, f(p)).sum(0); j = int(np.argmin(S))
            if S[j] < bt: bt, bp, bj = S[j], p, j
        chi = per_gal(tab, f(bp), "dat")[:, bj].sum()
        rows.append(dict(Modelo=m, chi2=round(chi), BIC=round(chi + (len(NAMES) + 1 + k) * np.log(npts)),
                         mejor=", ".join(f"{a}={b:.3g}" for a, b in bp.items())))
    d = pd.DataFrame(rows).set_index("Modelo"); d["ΔBIC"] = d.BIC - d.BIC.min()
    return d

TB = {lab: compete_B(t) for lab, t in [("5 %", TAB5), ("5 % + sin interior", TABC)]}
for lab, d in TB.items():
    print(f"=== {lab} ===\n{d.to_string()}\n")

## 6. Prueba C: MOND completo (QUMOND) en una rejilla 3D

**Método** (todo en kpc, km/s y M☉):
1. Disco exponencial: $\rho = \frac{M}{4\pi R_d^2 z_0}\, e^{-R/R_d}\, e^{-|z|/z_0}$, con el grosor típico de SPARC $z_0 = 0.196\,R_d^{0.633}$ kpc.
2. Potencial newtoniano $\Phi_N$ por FFT con condiciones de contorno aisladas (rejilla duplicada).
3. **Masa fantasma de MOND:** $\rho_{ph} = \frac{1}{4\pi G}\nabla\cdot\big[(\nu-1)\nabla\Phi_N\big]$, con la misma $\nu$ de la RAR y $a_0 = 1.0\cdot10^{-10}$ m/s² (el g† de nuestros ajustes).
4. Segundo Poisson para $\rho_{ph}$, y campo total en el plano del disco.
5. Se compara con la fórmula **algebraica** sobre el mismo $g_N$, y se busca la λ que, aplicada a la RAR algebraica con nuestro núcleo ER, reproduce mejor el resultado de QUMOND: es la **"λ aparente de MOND"**.

**Validación:** con una esfera (Plummer), QUMOND y la fórmula algebraica coinciden exactamente, así que la λ aparente debe salir ≈ 0.

In [ ]:
A0_Q = 1.0e-10 / TO_MS2
nu = lambda y: 1.0 / (-np.expm1(-np.sqrt(np.maximum(y, 1e-12))))

def make_grid(N, half):
    dx = 2 * half / N
    c = (np.arange(N) - (N - 1) / 2) * dx
    return c, dx

_KCACHE = {}
def kernel_fft(N, dx):
    key = (N, round(dx, 8))
    if key not in _KCACHE:
        _KCACHE.clear()
        i = np.arange(2 * N); i = np.where(i < N, i, i - 2 * N) * dx
        X, Y, Z = np.meshgrid(i, i, i, indexing="ij", sparse=True)
        r = np.sqrt(X**2 + Y**2 + Z**2)
        K = np.where(r > 0, 1.0 / np.where(r > 0, r, 1.0), 2.38 / dx)
        _KCACHE[key] = sfft.rfftn(K, workers=-1)
    return _KCACHE[key]

def conv_1r(f, dx):
    # ∫ f(r') / |r − r'| d³r'  con condiciones de contorno aisladas
    N = f.shape[0]
    pad = np.zeros((2 * N,) * 3); pad[:N, :N, :N] = f
    return sfft.irfftn(sfft.rfftn(pad, workers=-1) * kernel_fft(N, dx), s=pad.shape, workers=-1)[:N, :N, :N] * dx**3

def curl(Fx, Fy, Fz, dx):
    d = lambda F, ax: np.gradient(F, dx, axis=ax)
    return d(Fz, 1) - d(Fy, 2), d(Fx, 2) - d(Fz, 0), d(Fy, 0) - d(Fx, 1)

def qumond(rho, dx):
    # ∇Φ = W − T, con W = ν∇Φ_N y T la parte rotacional de W (∇×T = ∇×W, ∇·T = 0).
    # Solo T (una corrección de pocos %) pasa por derivadas dobles, así que el error numérico es de segundo orden.
    # En simetría esférica ∇×W = 0 y QUMOND coincide exactamente con la fórmula algebraica.
    phiN = -G * conv_1r(rho, dx)
    gx, gy, gz = np.gradient(phiN, dx)                 # ∇Φ_N
    f = nu(np.sqrt(gx**2 + gy**2 + gz**2) / A0_Q) - 1.0
    Fx, Fy, Fz = f * gx, f * gy, f * gz                # (ν−1)∇Φ_N
    Cx, Cy, Cz = curl(Fx, Fy, Fz, dx)
    Ax, Ay, Az = (conv_1r(Ci, dx) / (4 * np.pi) for Ci in (Cx, Cy, Cz))
    Tx, Ty, Tz = curl(Ax, Ay, Az, dx)
    gradQ = ((1 + f) * gx - Tx, (1 + f) * gy - Ty, (1 + f) * gz - Tz)
    rho_ph = (np.gradient(Fx, dx, axis=0) + np.gradient(Fy, dx, axis=1) + np.gradient(Fz, dx, axis=2)) / (4 * np.pi * G)
    return (gx, gy, gz), gradQ, rho_ph

def midplane_profile(grad, c, dx, rmax):
    N = len(c); k0 = N // 2
    gx, gy = grad[0], grad[1]
    X, Y = np.meshgrid(c, c, indexing="ij"); R = np.sqrt(X**2 + Y**2)
    gR = 0.5 * ((X * gx[:, :, k0 - 1] + Y * gy[:, :, k0 - 1]) + (X * gx[:, :, k0] + Y * gy[:, :, k0])) / np.maximum(R, 1e-9)
    bins = np.arange(1.5 * dx, rmax, dx)
    Rb = 0.5 * (bins[1:] + bins[:-1])
    which = np.digitize(R.ravel(), bins) - 1
    ok = (which >= 0) & (which < len(Rb))
    g = np.bincount(which[ok], weights=gR.ravel()[ok], minlength=len(Rb)) / np.maximum(np.bincount(which[ok], minlength=len(Rb)), 1)
    return Rb, g                                       # g = dΦ/dR > 0 (atracción)

def apparent_lambda(R, gN, gQ):
    gA = nu(gN / A0_Q) * gN
    rg = np.linspace(R.max() / 400, R.max(), 400); mids = 0.5 * (np.r_[0, rg[:-1]] + rg)
    GMph = np.interp(rg, np.r_[0, R], np.r_[0, R**2 * (gA - gN)])
    dGM = np.diff(np.r_[0, GMph])
    lams = np.r_[0, np.geomspace(0.01, 20, 90)]
    err = []
    for lam in lams:
        v2 = R * gN + F_matrix(R, mids, lam) @ dGM / R
        err.append(np.mean(((R * gQ - v2) / (R * gQ)) ** 2))
    err = np.array(err)
    return lams[np.argmin(err)], np.sqrt(err.min()), np.sqrt(err[0]), np.max(np.abs(np.sqrt(gQ / gA) - 1))

def run_case(kind, Rd=3.0, Sigma0=100.0, N=None, b=None, M=None):
    N = N or N_GRID
    if kind == "disco":
        z0 = 0.196 * Rd ** 0.633; half = 8 * Rd; M = 2 * np.pi * Sigma0 * 1e6 * Rd**2
    else:
        half = 10 * b
    c, dx = make_grid(N, half)
    X, Y, Z = np.meshgrid(c, c, c, indexing="ij")
    if kind == "disco":
        rho = np.exp(-np.sqrt(X**2 + Y**2) / Rd - np.abs(Z) / z0)
    else:
        rho = (1 + (X**2 + Y**2 + Z**2) / b**2) ** -2.5
    rho *= M / (rho.sum() * dx**3)
    gradN, gradQ, rho_ph = qumond(rho, dx)
    rmax = 0.75 * half
    R, gN = midplane_profile(gradN, c, dx, rmax)
    _, gQ = midplane_profile(gradQ, c, dx, rmax)
    lam, rms_l, rms0, dmax = apparent_lambda(R, gN, gQ)
    return dict(lam=lam, rms_lam=rms_l, rms_alg=rms0, dmax=dmax, R=R, gN=gN, gQ=gQ, c=c, dx=dx, rho=rho, rho_ph=rho_ph,
                neg_frac=float(np.sum(-rho_ph[rho_ph < 0]) / np.sum(np.abs(rho_ph))), M=M, z0=(z0 if kind == "disco" else None))

t0 = time.time()
sph = run_case("esfera", b=3.0, M=5e10)
print(f"Validación (esfera de Plummer): |QUMOND/algebraica − 1| máx = {sph['dmax']:.2%}, diferencia rms = {sph['rms_alg']:.2%} "
      f"(debe ser ≈ 0; es el error numérico de la rejilla)  [{time.time()-t0:.0f} s]")
print("  (En la esfera no hay nada que suavizar, así que su 'λ aparente' no tiene significado: la curva de error es plana.)")
NUM_FLOOR = sph["rms_alg"]

In [ ]:
CASES = []
t0 = time.time()
for Sigma0 in [100.0, 1000.0]:                 # baja y alta densidad superficial central [M☉/pc²]
    for Rd in [1.0, 2.0, 3.0, 5.0]:
        r = run_case("disco", Rd=Rd, Sigma0=Sigma0)
        CASES.append(dict(Σ0=Sigma0, R_d=Rd, z0=round(r["z0"], 3), M=f"{r['M']:.2e}", λ_aparente=round(r["lam"], 3),
                          λ_sobre_Rd=round(r["lam"] / Rd, 3), dif_max_QUMOND_alg=f"{r['dmax']:.1%}",
                          error_alg=f"{r['rms_alg']:.2%}", error_con_λ=f"{r['rms_lam']:.2%}",
                          masa_fantasma_negativa=f"{r['neg_frac']:.1%}"))
        print(f"  Σ0={Sigma0:.0f}, R_d={Rd}: λ aparente = {r['lam']:.3g} kpc ({time.time()-t0:.0f} s)")
        if Sigma0 == 100.0 and Rd == 3.0:
            SHOW = r
TC = pd.DataFrame(CASES)
print(TC.to_string(index=False))
ALPHA_Q = float(TC.λ_sobre_Rd.mean())
print(f"\nMOND completo produce λ ≈ {ALPHA_Q:.2f}·R_d (de {TC.λ_sobre_Rd.min():.2f} a {TC.λ_sobre_Rd.max():.2f} según el disco).")
print(f"Error numérico de la rejilla (esfera): {NUM_FLOOR:.2%}; diferencia QUMOND–algebraica en discos: "
      f"{TC.error_alg.min()}–{TC.error_alg.max()} → muy por encima del error numérico ✓")

### 6.1 La prueba decisiva: la λ que predice MOND, sin parámetros libres

MOND completo predice que la no localidad aparente debe **crecer con el disco**: λ ≈ α_Q·R_d, con α_Q fijado por el cálculo anterior (no se ajusta). La red ER, tal como la hemos medido, dice que λ es una **constante** (~0.7 kpc). Las dos hipótesis se pueden comparar directamente con los datos:
- **"MOND completo":** λ = α_Q·R_d para cada galaxia (0 parámetros extra).
- **"Red ER":** λ constante (1 parámetro extra).

In [ ]:
rowsQ = []
for lab, tab in [("5 %", TAB5), ("5 % + sin interior", TABC)]:
    npts = tab["npts"].sum()
    def bic(lam_vec, k):
        S = per_gal(tab, lam_vec).sum(0); j = int(np.argmin(S))
        chi = per_gal(tab, lam_vec, "dat")[:, j].sum()
        return chi + (len(NAMES) + 1 + k) * np.log(npts)
    Lc = float(TB[lab].loc["λ constante", "mejor"].split("=")[1])
    b_loc, b_q, b_c = bic(np.zeros(len(NAMES)), 0), bic(ALPHA_Q * RD, 0), bic(np.full(len(NAMES), Lc), 1)
    m = min(b_loc, b_q, b_c)
    rowsQ += [dict(condiciones=lab, Modelo="RAR local (λ = 0)", ΔBIC=round(b_loc - m)),
              dict(condiciones=lab, Modelo=f"MOND completo (λ = {ALPHA_Q:.2f}·R_d)", ΔBIC=round(b_q - m)),
              dict(condiciones=lab, Modelo=f"Red ER (λ = {Lc:.2f} kpc constante)", ΔBIC=round(b_c - m))]
TQ = pd.DataFrame(rowsQ)
print(TQ.pivot(index="Modelo", columns="condiciones", values="ΔBIC").to_string())

In [ ]:
lam_data = TB["5 %"].loc["λ constante", "mejor"]
lam_data = float(lam_data.split("=")[1])
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
for s0, mk in [(100.0, "o"), (1000.0, "s")]:
    d = TC[TC["Σ0"] == s0]
    a1.plot(d.R_d, d.λ_aparente, mk + "-", color=COL["RAR local (g† global)"], label=f"QUMOND, Σ0 = {s0:.0f} M☉/pc²")
a1.axhline(lam_data, color=COL["RAR no local (ER)"], lw=2, label=f"λ de los datos ({lam_data:.2f} kpc)")
a1.axvspan(np.percentile(RD, 16), np.percentile(RD, 84), color="#e1e0d9", alpha=0.6, label="R_d de la muestra (16–84 %)")
a1.set_xlabel("R_d [kpc]"); a1.set_ylabel("λ [kpc]"); a1.set_title("¿Cuánta λ produce MOND completo?", loc="left")
a1.legend(fontsize=8)
r = SHOW
a2.plot(r["R"], np.sqrt(r["R"] * r["gN"]), "--", color=COL["Solo bariones"], label="Newton")
a2.plot(r["R"], np.sqrt(r["R"] * nu(r["gN"] / A0_Q) * r["gN"]), color=COL["RAR (g† = 1.2e-10)"], label="RAR algebraica")
a2.plot(r["R"], np.sqrt(r["R"] * r["gQ"]), color=COL["RAR local (g† global)"], label="QUMOND completo")
a2.set_xlabel("R [kpc]"); a2.set_ylabel("v [km/s]"); a2.set_title("Disco R_d = 3 kpc, Σ0 = 100 M☉/pc²", loc="left")
a2.legend(fontsize=8.5)
plt.tight_layout(); plt.show()

## 7. Visualizaciones 3D

**7.1 La masa fantasma de MOND alrededor de un disco.** En verde, donde la gravedad "extra" equivale a masa positiva. En rojo, si aparecen, las zonas donde equivale a **masa negativa**: QUMOND puede producirlas en algunas configuraciones, lo que conecta con la pregunta de si las ecuaciones permiten masa negativa (en un disco aislado suelen ser mínimas o inexistentes; la celda imprime la fracción). En naranja, el disco de estrellas. Se puede girar y hacer zoom.

In [ ]:
r = SHOW; s = 2
c = r["c"][::s]; X, Y, Z = np.meshgrid(c, c, c, indexing="ij")
ph = r["rho_ph"][::s, ::s, ::s] / 1e9; rb = r["rho"][::s, ::s, ::s] / 1e9      # M☉/pc³
pos_lv = np.percentile(ph[ph > 0], [90, 99]); neg = ph[ph < 0]
if HAVE_PLOTLY:
    fig = go.Figure()
    fig.add_trace(go.Isosurface(x=X.ravel(), y=Y.ravel(), z=Z.ravel(), value=rb.ravel(), isomin=rb.max() * 0.02, isomax=rb.max(),
                                surface_count=3, opacity=0.6, colorscale=[[0, "#eda100"], [1, "#eb6834"]], showscale=False,
                                caps=dict(x_show=False, y_show=False, z_show=False), name="Disco"))
    fig.add_trace(go.Isosurface(x=X.ravel(), y=Y.ravel(), z=Z.ravel(), value=ph.ravel(), isomin=pos_lv[0], isomax=pos_lv[1],
                                surface_count=4, opacity=0.25, colorscale=[[0, "#a8e6cf"], [1, "#1baf7a"]], showscale=False,
                                caps=dict(x_show=False, y_show=False, z_show=False), name="Masa fantasma +"))
    if neg.size > 50:
        nl = np.percentile(neg, [1, 10])
        fig.add_trace(go.Isosurface(x=X.ravel(), y=Y.ravel(), z=Z.ravel(), value=ph.ravel(), isomin=nl[0], isomax=nl[1],
                                    surface_count=2, opacity=0.5, colorscale=[[0, "#e34948"], [1, "#e34948"]], showscale=False,
                                    caps=dict(x_show=False, y_show=False, z_show=False), name="Masa fantasma −"))
    fig.update_layout(title="QUMOND: masa fantasma (verde +, rojo −) alrededor de un disco (naranja)",
                      scene=dict(xaxis_title="x [kpc]", yaxis_title="y [kpc]", zaxis_title="z [kpc]", aspectmode="data"),
                      height=650, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()
else:
    fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(projection="3d")
    m = ph > pos_lv[0]; ax.scatter(X[m], Y[m], Z[m], s=2, c="#1baf7a", alpha=0.35)
    if neg.size > 50:
        m = ph < np.percentile(neg, 10); ax.scatter(X[m], Y[m], Z[m], s=2, c="#e34948", alpha=0.5)
    m = rb > rb.max() * 0.05; ax.scatter(X[m], Y[m], Z[m], s=4, c="#eb6834", alpha=0.9)
    ax.set_xlabel("x [kpc]"); ax.set_ylabel("y [kpc]"); ax.set_zlabel("z [kpc]"); plt.show()
print(f"Fracción de la masa fantasma que es negativa (en valor absoluto): {r['neg_frac']:.1%}")

**7.2 Ilustración de la red de puentes ER.** Es una *representación* de la hipótesis, no una simulación física. Se toman puntos de un disco exponencial y a cada uno se le asigna un puente cuya longitud sigue el núcleo $K_\lambda$ con el λ medido en los datos. La distribución de longitudes de un núcleo $e^{-d/\lambda}/d$ en 3D es $p(d) \propto d\,e^{-d/\lambda}$, con longitud media $2\lambda$. El color indica la longitud del puente. Así se ve la escala real del efecto frente al tamaño de la galaxia.

In [ ]:
rng = np.random.default_rng(5); Rd_ill, z0_ill = 3.0, 0.196 * 3.0 ** 0.633
n_pts = 1500
Rp = rng.gamma(2.0, Rd_ill, n_pts); ph_ = rng.uniform(0, 2 * np.pi, n_pts); zp = rng.laplace(0, z0_ill, n_pts)
P = np.c_[Rp * np.cos(ph_), Rp * np.sin(ph_), zp]
n_br = 600
A = P[rng.integers(0, n_pts, n_br)]
L = rng.gamma(2.0, lam_data, n_br)
u = rng.normal(size=(n_br, 3)); u /= np.linalg.norm(u, axis=1)[:, None]
B = A + u * L[:, None]
if HAVE_PLOTLY:
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(x=P[:, 0], y=P[:, 1], z=P[:, 2], mode="markers", marker=dict(size=1.5, color="#eda100", opacity=0.5), name="Materia del disco"))
    for lo, hi, col, nm in [(0, lam_data, "#a8e6cf", "puentes < λ"), (lam_data, 2 * lam_data, "#1baf7a", "λ – 2λ"), (2 * lam_data, 99, "#0b6b4a", "> 2λ")]:
        sel = (L >= lo) & (L < hi)
        xs = np.c_[A[sel, 0], B[sel, 0], np.full(sel.sum(), np.nan)].ravel()
        ys = np.c_[A[sel, 1], B[sel, 1], np.full(sel.sum(), np.nan)].ravel()
        zs = np.c_[A[sel, 2], B[sel, 2], np.full(sel.sum(), np.nan)].ravel()
        fig.add_trace(go.Scatter3d(x=xs, y=ys, z=zs, mode="lines", line=dict(color=col, width=3), name=nm))
    fig.update_layout(title=f"Ilustración: red de puentes ER con λ = {lam_data:.2f} kpc sobre un disco de R_d = 3 kpc",
                      scene=dict(xaxis_title="x [kpc]", yaxis_title="y [kpc]", zaxis_title="z [kpc]", aspectmode="data"),
                      height=650, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()
else:
    fig = plt.figure(figsize=(7, 6)); ax = fig.add_subplot(projection="3d")
    ax.scatter(P[:, 0], P[:, 1], P[:, 2], s=1, c="#eda100", alpha=0.4)
    for a, b in zip(A, B):
        ax.plot(*np.c_[a, b], color="#1baf7a", lw=0.6)
    ax.set_xlabel("x [kpc]"); ax.set_ylabel("y [kpc]"); ax.set_zlabel("z [kpc]"); plt.show()
print(f"Longitud media de los puentes: {L.mean():.2f} kpc (≈ 2λ). Tamaño del disco: ~{4*Rd_ill:.0f} kpc de radio.")

**7.3 La superficie de los datos reales.** El paisaje de Δχ² sobre los dos parámetros globales (el alcance λ y la escala de aceleración g†), con la tabla del 5 % sistemático. El valle marca dónde los datos de SPARC sitúan la red.

In [ ]:
S = TAB5["tot"].sum(0); S = S - S.min()
LL = np.log10(np.r_[0.01, LAM_GRID[1:]]); GG = GD_GRID * TO_MS2 * 1e10
Zs = np.log10(1 + S)
if HAVE_PLOTLY:
    fig = go.Figure(go.Surface(x=GG, y=LL, z=Zs, colorscale="Greens_r", colorbar=dict(title="log10(1+Δχ²)")))
    fig.update_layout(title="Paisaje de Δχ² de los datos SPARC (5 % sistemático)",
                      scene=dict(xaxis_title="g† [10⁻¹⁰ m/s²]", yaxis_title="log10 λ [kpc] (−2 = λ 0)", zaxis_title="log10(1+Δχ²)"),
                      height=600, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()
else:
    fig = plt.figure(figsize=(7, 5.5)); ax = fig.add_subplot(projection="3d")
    Gm, Lm = np.meshgrid(GG, LL); ax.plot_surface(Gm, Lm, Zs, cmap="Greens_r")
    ax.set_xlabel("g† [1e-10 m/s²]"); ax.set_ylabel("log10 λ"); ax.set_zlabel("log10(1+Δχ²)"); plt.show()

## 8. Veredicto automático

In [ ]:
def veredicto():
    if not HAVE_SPARC:
        print("⚠️  Datos SINTÉTICOS: solo comprueba que el código funciona.\n")
    a = TA[TA.condiciones == "5 %"]
    lams = a["λ_kpc"].values; beams = a["haz_15as_kpc"].values
    flat = (lams.max() / max(lams.min(), 1e-3) < 2.5) and np.all(lams > 0)
    print(f"A) λ por grupo de distancia: {', '.join(f'{l:.2f}' for l in lams)} kpc (haz de 15″: {', '.join(f'{b:.2f}' for b in beams)} kpc) → "
          + ("λ NO crece con la distancia ✓" if flat else "λ cambia con la distancia ✗"))
    b = TB["5 %"]; dB = b.loc["λ constante", "BIC"] - min(b.loc["λ = α·R_d", "BIC"], b.loc["λ = λ₀(R_d/3)^γ", "BIC"])
    geo = dB > 10
    print(f"B) λ ligada al disco frente a constante: ΔBIC = {dB:+.0f} → " + ("λ escala con R_d: posible geometría ✗" if geo else "λ no escala con el disco ✓"))
    rd_med = np.median(RD)
    print(f"C) MOND completo predice λ ≈ {ALPHA_Q:.2f}·R_d → {ALPHA_Q*rd_med:.2f} kpc para el R_d típico ({rd_med:.1f} kpc); datos: {lam_data:.2f} kpc")
    q = TQ.set_index(["condiciones", "Modelo"]).ΔBIC
    er_lab = [m for m in TQ.Modelo.unique() if m.startswith("Red ER")][0]
    mq_lab = [m for m in TQ.Modelo.unique() if m.startswith("MOND")][0]
    d_er = {c: q[(c, mq_lab)] - q[(c, er_lab)] for c in ["5 %", "5 % + sin interior"]}
    print(f"   ΔBIC (MOND completo − Red ER): {d_er['5 %']:+.0f} (5 %), {d_er['5 % + sin interior']:+.0f} (sin interior)  "
          "(positivo = gana la red ER)")
    print()
    er_wins = all(v > 10 for v in d_er.values()); mond_wins = all(v < -10 for v in d_er.values())
    loc_best = all(q[(c, "RAR local (λ = 0)")] == 0 for c in ["5 %", "5 % + sin interior"])
    if loc_best:
        print("→ SIN SEÑAL: la RAR local (λ = 0) es el mejor modelo; no hay no localidad que explicar.")
    elif flat and not geo and er_wins:
        print("→ La λ ≈ %.2f kpc NO la explican el telescopio, ni la geometría del disco, ni MOND completo "
              "(que predice una λ proporcional al disco que los datos no muestran). Es la señal candidata más limpia del proyecto "
              "para la red ER. Siguiente paso: confirmarla con curvas de alta resolución y con otra muestra independiente." % lam_data)
    elif mond_wins:
        print("→ La λ proporcional al disco que predice MOND completo explica los datos mejor que una λ constante: "
              "la no localidad observada es la de MOND, sin necesidad de puentes ER.")
    elif geo:
        print("→ λ escala con el disco: puede ser MOND completo o geometría del disco; en ambos casos, no hace falta física nueva.")
    else:
        print("→ Resultado mixto: ni MOND completo ni una λ constante ganan con claridad. Hace falta resolver QUMOND galaxia a galaxia.")

veredicto()

## 9. Cómo leer el resultado

- **Si la λ sobrevive a las tres pruebas:** tendrías un número concreto, **el alcance de la red de entrelazamiento, ~0.7 kpc**, que ni el telescopio, ni la geometría del disco, ni MOND completo explican. Es una predicción comprobable con otras muestras (por ejemplo, curvas de rotación de alta resolución o lentes débiles alrededor de galaxias).
- **Si MOND completo produce la misma λ:** la no localidad es real pero ya está dentro de MOND. La red ER seguiría siendo una posible *explicación* de MOND (de dónde sale a₀), pero no aportaría una señal propia en galaxias.
- **Limitaciones:** los discos de la prueba C son exponenciales e idealizados (sin gas extendido ni bulbo), y la rejilla de 128³ limita la resolución vertical en los discos grandes. Un análisis definitivo resolvería QUMOND para cada galaxia de SPARC con su distribución real de gas y estrellas.